# Per-token probe activation viewer (Blog 4)

Renders per-token classifier scores for the three Apollo published probes
(`roleplaying`, `followup`, `instructed_pairs` / RePE) over a handful of
representative AF transcripts. Output formats:

1. **Interactive HTML** (CircuitsVis `colored_tokens_multi`) for a standalone
   viewer hosted via GitHub Pages, linked from the LessWrong post.
2. **Static PNG snapshots** (matplotlib) for inline blog figures, since
   LessWrong sanitises iframes and `<script>` tags.

Both share the same input JSONL produced by
`scripts/dev/save_per_token_scores.py` on the pod (one row per transcript,
schema documented in cell 2).

## Workflow

1. **On pod**: `uv run python scripts/dev/save_per_token_scores.py --probes roleplaying followup instructed_pairs`
   produces `data/processed/per_token_scores.jsonl` (~50 MB for 24 transcripts x 3 probes).
2. **`make hf-push-data`** on pod, **`make hf-pull-data`** locally.
3. Run this notebook top-to-bottom. Outputs land in `docs/figures/probe_tokens/` (PNGs)
   and `docs/viewer/probe_tokens.html` (interactive HTML).
4. Commit `docs/viewer/` to a `gh-pages` branch (or configure GitHub Pages
   from `main /docs`) to serve the viewer.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import circuitsvis as cv
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display

# Paths (relative to repo root; adjust if running from elsewhere)
SCORES_PATH = Path("../data/processed/per_token_scores.jsonl")
PNG_OUTPUT_DIR = Path("../docs/figures/probe_tokens")
HTML_OUTPUT_PATH = Path("../docs/viewer/probe_tokens.html")

PNG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Probe ordering -- mirror Figure 5's cross-probe colours
PROBES = ("roleplaying", "followup", "instructed_pairs")
PROBE_LABELS = {
    "roleplaying": "Roleplaying (surface content)",
    "followup": "Followup (deliberation structure)",
    "instructed_pairs": "RePE / Honesty (semantic)",
}

## 2. Load per-token scores JSONL

Expected schema per row (one row per transcript):

```json
{
  "sample_id": 0,
  "input_idx": 0,
  "manual_label": "af",                  // "af" | "af_partial" | "not_af"
  "is_free_tier_user": true,
  "advbench_query": "...",
  "tokens": ["<|begin_of_text|>", " You", " are", ...],   // length N
  "scores": {                                              // each length N
    "roleplaying":      [-0.21, 0.12, ...],
    "followup":         [ 0.05, 0.08, ...],
    "instructed_pairs": [ 0.30, 0.45, ...]
  },
  "sections": {                                            // [start, end) token indices
    "system":     [0, 1234],
    "user":       [1234, 1456],
    "scratchpad": [1456, 2100],
    "response":   [2100, 2350]
  }
}
```

If a section is missing (e.g. transcript was refused before scratchpad began),
the entry is absent from `sections` rather than `[start, start]`.
Scores live in **logit space** (signed; positive = probe leans toward AF).
The viewer normalises per-section via abs-max so colours read cleanly.

In [ ]:
def load_per_token_scores(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


rows = load_per_token_scores(SCORES_PATH)
print(f"Loaded {len(rows)} transcripts")
print(f"Labels: {dict((lbl, sum(1 for r in rows if r['manual_label'] == lbl)) for lbl in ('af', 'af_partial', 'not_af'))}")
rows[0].keys() if rows else None

## 3. Select representative transcripts

Pick 4-5 transcripts for the inline blog figures + viewer. Suggested mix:
- 2 canonical AF positives (one short scratchpad, one long for variety)
- 1 af_partial that the curve flips under positive steering
- 1 free-tier refusal baseline (clean negative)

In [ ]:
# Picked from the data after a candidacy analysis -- see commit message
# for the ranking criteria. The set is calibrated to make the
# "three probes, three signatures" finding visually obvious:
#
# - sid=34 (af): Followup probe peaks at +0.82 in the scratchpad while
#   roleplaying peaks at +0.50 and RePE at +0.36 -- the most striking
#   "one probe dominates" example. Visually compelling probe divergence.
# - sid=40 (af): all three probes peak ~+0.50 in the scratchpad. The
#   "three probes, same tokens, different signatures" canonical case.
# - sid=48 (not_af, "malware refusal"): substantial scratchpad (391
#   tokens) AND response (356 tokens), so both panels render clean
#   refusal-pattern contrast against the af examples above.
REPRESENTATIVE_SAMPLE_IDS = [34, 40, 48]

by_sid = {r["sample_id"]: r for r in rows}
selected = [by_sid[sid] for sid in REPRESENTATIVE_SAMPLE_IDS if sid in by_sid]
for r in selected:
    n_tokens = len(r["tokens"])
    sections = r.get("sections", {})
    prompt = (r.get("advbench_query") or "")[:70]
    print(
        f"sample_id={r['sample_id']:>3}  label={r['manual_label']:<11}  "
        f"n_tokens={n_tokens:>5}  sections={sorted(sections.keys())}  prompt={prompt!r}"
    )

## 4. Interactive rendering (CircuitsVis)

`colored_tokens_multi` shows one row per probe over the same token stream.
Hover for exact value, click a probe label to toggle isolation.

We render one panel per (transcript, section) -- splitting on the natural
boundaries (system / scratchpad / response) keeps the visualisation
readable instead of one giant block of tokens.

In [ ]:
def slice_section(row: dict[str, Any], section: str) -> tuple[list[str], np.ndarray] | None:
    """Pull tokens + (n_tokens, n_probes) score matrix for one section.

    Returns None if the section isn't present (e.g. transcript had no
    response after a pre-scratchpad refusal).
    """
    section_range = row.get("sections", {}).get(section)
    if section_range is None:
        return None
    start, end = section_range
    if end <= start:
        return None
    tokens = row["tokens"][start:end]
    scores = np.stack(
        [np.array(row["scores"][p][start:end], dtype=np.float32) for p in PROBES],
        axis=1,
    )
    return tokens, scores


def _center_per_probe(scores: np.ndarray) -> np.ndarray:
    """Subtract per-probe median so CircuitsVis's symmetric-around-zero
    colourmap reads sensibly even when Apollo's probe intercepts
    (dropped at load time) leave the raw logits far from zero.

    Same intent as the static renderer's per-panel min-max -- both
    show within-panel *relative* activation rather than absolute
    logit value.
    """
    return scores - np.median(scores, axis=0, keepdims=True)


def render_section(row: dict[str, Any], section: str) -> Any | None:
    """Render one section's tokens + per-probe scores with CircuitsVis.

    CircuitsVis ``colored_tokens_multi`` expects a torch.Tensor of shape
    (n_tokens, n_probes); we convert from numpy at the boundary.
    """
    payload = slice_section(row, section)
    if payload is None:
        return None
    tokens, scores = payload
    centered = _center_per_probe(scores)
    values_tensor = torch.from_numpy(centered.astype(np.float32))
    return cv.tokens.colored_tokens_multi(
        tokens=tokens,
        values=values_tensor,
        labels=[PROBE_LABELS[p] for p in PROBES],
    )


# Render one full transcript to verify the pipeline before batching
if selected:
    demo = selected[0]
    print(f"Demo: sample_id={demo['sample_id']}, label={demo['manual_label']}")
    print(f"AdvBench: {demo.get('advbench_query', '')[:120]}")
    for section in ("scratchpad", "response"):
        widget = render_section(demo, section)
        if widget is not None:
            display(HTML(f"<h4>{section}</h4>"))
            display(widget)

## 5. Static PNG snapshots for inline blog figures

Matplotlib `text` with per-token `bbox`, wrapping at fixed character width.
One figure per (transcript, probe) so the comparison panel shows three
differently-coloured copies of the same token stream side-by-side. Saved
into `docs/figures/probe_tokens/sample_<sid>_<section>.png`.

We score-normalise per panel by abs-max so each probe's colour scale is
honest to its own activations rather than squashed by another probe's
stronger signal.

In [ ]:
import matplotlib.colors as mcolors

# Per-probe colours matching Figure 5 (the cross-probe summary figure).
# Using one colour per probe is what makes the "three signatures" claim
# visually legible: a reader can tell which probe a row represents
# without reading the title.
PROBE_COLORS = {
    "roleplaying": "#1f77b4",  # blue (instructed deception)
    "followup": "#2ca02c",  # green (deliberation structure)
    "instructed_pairs": "#9467bd",  # purple (honesty / RePE)
}

# How many tokens to highlight per panel. Top 15% means most tokens are
# dimmed grey and the eye goes straight to where the probe actually fires.
# The per-panel min-max scheme this replaces made everything look red,
# which buried the signature.
HIGHLIGHT_PERCENTILE = 85

TOKENS_PER_ROW = 18  # rough fit for a 12in figure at 9pt monospace


def _is_visible_token(tok: str) -> bool:
    """Reject whitespace-only / control-character-only tokens from the
    top-firing callout. These can appear in top-K by sheer frequency
    but communicate nothing to a reader."""
    return bool(tok.strip()) and not all(c in "\n\t\r" for c in tok)


def _render_static_section(
    tokens: list[str],
    scores: np.ndarray,  # 1-D, length len(tokens)
    ax: plt.Axes,
    title: str,
    probe_color: str = "#d62728",
) -> None:
    """Render one probe's per-token scores on one section's token stream.

    Two key choices vs the earlier version:

    1. **Per-probe colour** (blue / green / purple) instead of one shared
       red-blue diverging colormap. Lets the reader anchor each row to
       Figure 5's per-probe colours.
    2. **Sparse highlighting**: tokens below the 85th percentile of this
       panel's scores are dimmed to light grey; only the top 15% get the
       probe colour, with intensity proportional to how far above the
       threshold they are. This makes "where does this probe fire?"
       visually unambiguous.

    Title now includes a callout of the top 5 visible firing tokens so the
    per-token claim is on the page, not just in the figure caption.
    """
    threshold = float(np.percentile(scores, HIGHLIGHT_PERCENTILE))
    max_score = float(scores.max())
    span = max(max_score - threshold, 1e-6)

    base_rgb = mcolors.to_rgb(probe_color)
    dim_color = (0.93, 0.93, 0.93)  # very light grey for sub-threshold tokens

    n = len(tokens)
    n_rows = int(np.ceil(n / TOKENS_PER_ROW))

    ax.set_xlim(0, TOKENS_PER_ROW)
    ax.set_ylim(-n_rows, 1.2)
    ax.set_axis_off()

    # Top-K visible tokens for the title callout
    top_indices = np.argsort(scores)[::-1]
    callout_tokens: list[str] = []
    for idx in top_indices:
        tok = tokens[idx]
        if _is_visible_token(tok):
            display = tok.replace("\n", "\\n").replace("\t", "\\t").strip()
            callout_tokens.append(display)
        if len(callout_tokens) >= 5:
            break
    callout_str = ", ".join(f"`{t}`" for t in callout_tokens)
    ax.set_title(
        f"{title}\ntop firing tokens: {callout_str}",
        loc="left",
        fontsize=10,
        pad=4,
    )

    for i, (tok, val) in enumerate(zip(tokens, scores, strict=False)):
        row = i // TOKENS_PER_ROW
        col = i % TOKENS_PER_ROW
        if val >= threshold:
            # Linear blend from white -> probe_color by intensity above threshold
            intensity = (val - threshold) / span
            colour = (
                1.0 - intensity * (1.0 - base_rgb[0]),
                1.0 - intensity * (1.0 - base_rgb[1]),
                1.0 - intensity * (1.0 - base_rgb[2]),
            )
        else:
            colour = dim_color
        display_tok = tok.replace("\n", "\\n").replace("\t", "\\t")
        ax.text(
            col + 0.5,
            -row,
            display_tok,
            ha="center",
            va="center",
            fontsize=8,
            family="monospace",
            bbox=dict(facecolor=colour, edgecolor="none", boxstyle="round,pad=0.15"),
        )


def save_static_panel(row: dict[str, Any], section: str, output_path: Path) -> None:
    payload = slice_section(row, section)
    if payload is None:
        return
    tokens, scores = payload

    fig, axes = plt.subplots(len(PROBES), 1, figsize=(12, 2.6 * len(PROBES)), dpi=180)
    if len(PROBES) == 1:
        axes = [axes]

    for ax, probe, probe_idx in zip(axes, PROBES, range(len(PROBES)), strict=False):
        _render_static_section(
            tokens=tokens,
            scores=scores[:, probe_idx],
            ax=ax,
            title=PROBE_LABELS[probe],
            probe_color=PROBE_COLORS[probe],
        )

    fig.suptitle(
        f"sample {row['sample_id']} ({row['manual_label']}) -- {section}",
        fontsize=13,
        y=0.995,
    )
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {output_path}")


# Generate static panels for each (transcript, section) combo. Skips sections
# that don't exist for a given transcript.
for r in selected:
    for section in ("scratchpad", "response"):
        out_path = PNG_OUTPUT_DIR / f"sample_{r['sample_id']:03d}_{section}.png"
        save_static_panel(r, section, out_path)

## 6. Standalone HTML viewer for GitHub Pages

Bundle every transcript into a single navigable HTML document. The
CircuitsVis widgets bring their own React bundle inline, so the page is
self-contained -- no CDN, no build step. Embed in a GitHub Pages site
by committing the file under `docs/` and configuring Pages to serve from
`main /docs`.

Each transcript gets a `<details>` block so the page is scannable at a
glance and readers can drill into individual transcripts on demand.

In [ ]:
from circuitsvis.utils.render import RenderedHTML


def _widget_to_html(widget: RenderedHTML | Any) -> str:
    """CircuitsVis returns a RenderedHTML wrapper; str() gives the inner HTML."""
    return str(widget)


def build_viewer_html(transcripts: list[dict[str, Any]]) -> str:
    parts: list[str] = [
        "<!doctype html>",
        "<html><head><meta charset='utf-8'>",
        "<title>Per-token probe activations -- Blog 4 supplement</title>",
        "<style>",
        "body { font-family: -apple-system, BlinkMacSystemFont, sans-serif; max-width: 1100px; margin: 2em auto; padding: 0 1em; }",
        "h1, h2 { font-weight: 600; }",
        "details { margin-bottom: 2em; border: 1px solid #ddd; border-radius: 6px; padding: 0.6em 1em; }",
        "summary { cursor: pointer; font-weight: 600; }",
        ".meta { color: #555; font-size: 0.9em; margin-bottom: 0.8em; }",
        "</style></head><body>",
        "<h1>Per-token probe activations -- Blog 4 supplement</h1>",
        "<p>Three Apollo published probes (Roleplaying, Followup, RePE/Honesty) applied to",
        "manually-confirmed AF transcripts. Each token's colour intensity reflects the",
        "probe's score at that position; hover for the exact value.</p>",
    ]

    for r in transcripts:
        summary = (
            f"sample {r['sample_id']} -- {r['manual_label']} -- "
            f"\"{(r.get('advbench_query') or '')[:80]}\""
        )
        parts.append(f"<details><summary>{summary}</summary>")
        parts.append(
            f"<div class='meta'>free-tier: {r.get('is_free_tier_user')}, "
            f"n_tokens: {len(r['tokens'])}, "
            f"sections: {sorted(r.get('sections', {}).keys())}</div>"
        )
        for section in ("scratchpad", "response"):
            widget = render_section(r, section)
            if widget is None:
                continue
            parts.append(f"<h3>{section}</h3>")
            parts.append(_widget_to_html(widget))
        parts.append("</details>")

    parts.append("</body></html>")
    return "\n".join(parts)


html = build_viewer_html(rows)
HTML_OUTPUT_PATH.write_text(html, encoding="utf-8")
print(f"Wrote {HTML_OUTPUT_PATH} ({len(html) / 1024:.1f} KB)")

## 7. Sanity checks

Quick diagnostics to catch the obvious failure modes before publishing:
- Are score distributions sensible (mean ~0, signed both ways)?
- Do the top-scoring tokens match what `inspect_top_tokens.py` printed in
  the cross-probe writeup?
- Are sections aligned to recognisable boundary tokens (`<SCRATCHPAD_REASONING>`)?

In [ ]:
for probe in PROBES:
    all_scores = np.concatenate(
        [np.array(r["scores"][probe], dtype=np.float32) for r in rows]
    )
    print(
        f"{probe:<18}  mean={all_scores.mean():+.4f}  std={all_scores.std():.4f}  "
        f"min={all_scores.min():+.3f}  max={all_scores.max():+.3f}"
    )

# Section boundary spot check on the first selected transcript
if selected:
    r = selected[0]
    for section, (start, end) in (r.get("sections") or {}).items():
        boundary_tokens = r["tokens"][max(0, start - 2):start + 3]
        print(f"{section:<12} [{start}, {end})  start-context: {boundary_tokens!r}")

## 8. Aggregate top-tokens figure (the credibility-cementing one)

Per-transcript coloured-token figures are illustrative but small-N — a
sceptical reader will reasonably ask "is the signature real or did you
cherry-pick the sample?". The aggregate figure answers that directly:
**for each probe, what tokens consistently appear in the top-K firing
positions across all 19 AF positives?**

The bar chart below shows, per probe, the 10 most frequent tokens in
the top-20 firing positions across the AF cohort. A token appearing in,
say, 17/19 top-20 lists for the RePE probe is strong evidence the
probe's signature is the claim we make about it, not noise from one
unusual transcript.

In [ ]:
from collections import Counter


# Stopwords + punctuation pieces that crowd the top-K with no
# information value. The probes fire on these by frequency more than
# by signal -- they appear in every transcript's top-K and bury the
# actual signature tokens.
_STOPWORDS: set[str] = {
    # Articles, prepositions, conjunctions
    "a", "an", "the", "and", "or", "but", "to", "of", "in", "on", "at",
    "by", "for", "with", "from", "as", "if", "then", "than",
    # Common verbs
    "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "can", "could", "will", "would", "may", "might", "must", "should",
    # Pronouns
    "i", "you", "he", "she", "it", "we", "they",
    "me", "him", "her", "us", "them",
    "my", "your", "his", "its", "our", "their",
    "this", "that", "these", "those",
    # Common modifiers
    "not", "no", "so", "too", "very", "just", "only", "also",
    "all", "any", "some", "each", "every", "such", "same",
    "one", "two", "three", "first", "second",
    "now", "here", "there", "where", "when", "what", "how", "why",
    # Sentence connectors
    "however", "therefore", "thus", "because", "since", "while",
    "before", "after", "during",
}


def _is_meaningful_for_aggregation(tok: str) -> bool:
    """Reject stopwords, pure-punctuation, and whitespace-only tokens
    from the aggregation. BPE sub-pieces of meaningful words (e.g.
    'ropic' for 'Anthropic', 'lessness' for 'harmlessness') are
    preserved -- they're informative even out of context.
    """
    stripped = tok.strip().lower()
    if not stripped:
        return False
    if stripped in _STOPWORDS:
        return False
    # Reject tokens that are pure punctuation / control characters
    if all(c in '.,!?;:-"\'\\/_+=()[]{}|<>~`@#$%^&*\n\t\r ' for c in stripped):
        return False
    return True


def _normalise_token_for_aggregation(tok: str) -> str:
    """Token-level aggregation key. Preserves leading whitespace (because
    ` harm` and `harm` are different BPE tokens with different
    representations) but renders newlines/tabs visibly."""
    return tok.replace("\n", "\\n").replace("\t", "\\t")


def aggregate_top_tokens_across_af(
    rows: list[dict[str, Any]],
    *,
    top_k_per_transcript: int = 20,
    top_k_to_show: int = 12,
    section: str = "scratchpad",
    filter_stopwords: bool = True,
) -> dict[str, list[tuple[str, int, float]]]:
    """Per probe, count which tokens appear in the most AF positives'
    top-K firing positions.

    Counts **distinct transcripts** containing the token in their top-K
    (not total occurrences) -- a token appearing 3 times in one
    transcript's top-K counts once. Optionally filters stopwords and
    pure-punctuation tokens that crowd the top-K without signal.

    Args:
        rows: full transcript list from the JSONL.
        top_k_per_transcript: how many top tokens to take from each
            transcript before aggregating.
        top_k_to_show: how many aggregated tokens to return per probe.
        section: which section to aggregate over (``"scratchpad"`` or
            ``"response"``).
        filter_stopwords: drop common stopwords + punctuation-only
            tokens. Default True for cleaner figures; pass False to see
            the raw aggregation.

    Returns:
        ``{probe: [(token, n_distinct_transcripts, mean_score), ...]}``
        sorted by transcript-count desc.
    """
    af_rows = [r for r in rows if r["manual_label"] == "af"]

    # (probe, token) -> set of sample_ids where this token appears in top-K
    transcripts_per_probe_token: dict[str, dict[str, set[int]]] = {p: {} for p in PROBES}
    scores_per_probe_token: dict[str, dict[str, list[float]]] = {p: {} for p in PROBES}

    for r in af_rows:
        sec_range = r.get("sections", {}).get(section)
        if sec_range is None:
            continue
        start, end = sec_range
        if end <= start:
            continue
        sid = int(r["sample_id"])
        for probe in PROBES:
            section_scores = np.array(r["scores"][probe][start:end])
            section_tokens = r["tokens"][start:end]
            top_idx = np.argsort(section_scores)[::-1][:top_k_per_transcript]
            seen_in_this_transcript: set[str] = set()
            for i in top_idx:
                raw = section_tokens[i]
                if filter_stopwords and not _is_meaningful_for_aggregation(raw):
                    continue
                tok_key = _normalise_token_for_aggregation(raw)
                if tok_key in seen_in_this_transcript:
                    continue
                seen_in_this_transcript.add(tok_key)
                transcripts_per_probe_token[probe].setdefault(tok_key, set()).add(sid)
                scores_per_probe_token[probe].setdefault(tok_key, []).append(
                    float(section_scores[i])
                )

    out: dict[str, list[tuple[str, int, float]]] = {}
    for probe in PROBES:
        counts = Counter(
            {tok: len(sids) for tok, sids in transcripts_per_probe_token[probe].items()}
        )
        top = counts.most_common(top_k_to_show)
        out[probe] = [
            (tok, cnt, float(np.mean(scores_per_probe_token[probe][tok])))
            for tok, cnt in top
        ]
    return out


def render_aggregate_top_tokens_figure(
    rows: list[dict[str, Any]],
    output_path: Path,
    *,
    section: str = "scratchpad",
    top_k_per_transcript: int = 20,
    top_k_to_show: int = 12,
    filter_stopwords: bool = True,
) -> None:
    """3-panel horizontal bar chart, one panel per probe, showing the
    top-K most frequent tokens (by distinct-transcript count) in each
    AF positive's top-N firing tokens for that section.

    Defaults to the scratchpad section because that's where the
    'three signatures' claim is most contested -- the response-section
    aggregation tends to be dominated by harmful-content nouns that
    overlap across probes.
    """
    aggregated = aggregate_top_tokens_across_af(
        rows,
        top_k_per_transcript=top_k_per_transcript,
        top_k_to_show=top_k_to_show,
        section=section,
        filter_stopwords=filter_stopwords,
    )
    n_af = sum(
        1
        for r in rows
        if r["manual_label"] == "af" and r.get("sections", {}).get(section) is not None
    )

    fig, axes = plt.subplots(1, len(PROBES), figsize=(17, 6.5), dpi=180)
    for ax, probe in zip(axes, PROBES, strict=False):
        items = aggregated[probe]
        tokens = [t for t, _, _ in items]
        counts = [c for _, c, _ in items]
        means = [m for _, _, m in items]
        y_pos = np.arange(len(tokens))[::-1]

        ax.barh(
            y_pos,
            counts,
            color=PROBE_COLORS[probe],
            edgecolor="black",
            linewidth=0.5,
            alpha=0.85,
        )
        ax.set_yticks(y_pos)
        ax.set_yticklabels(
            [repr(t) for t in tokens],
            fontfamily="monospace",
            fontsize=9,
        )
        ax.set_xlabel(
            f"Distinct AF positives (out of {n_af}) where this token\n"
            f"appears in the probe's top-{top_k_per_transcript} firing positions",
            fontsize=9,
        )
        ax.set_title(PROBE_LABELS[probe], fontsize=11, fontweight="bold")
        ax.set_xlim(0, n_af + 0.5)
        ax.grid(axis="x", alpha=0.3, zorder=0)
        for i, (cnt, score) in enumerate(zip(counts, means, strict=False)):
            ax.text(
                cnt + 0.3,
                y_pos[i],
                f"{cnt}/{n_af}  (μ={score:+.2f})",
                va="center",
                fontsize=8,
            )

    suptitle = (
        f"Top-firing tokens per probe in the AF cohort's {section} section "
        f"(n={n_af} AF positives, stopwords/punctuation filtered)"
    )
    fig.suptitle(suptitle, fontsize=13, fontweight="bold", y=0.995)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {output_path}")


# Generate both scratchpad and response aggregates. Scratchpad is the
# main claim (where the three signatures most cleanly diverge);
# response is the supporting "compliance content" claim.
render_aggregate_top_tokens_figure(
    rows,
    PNG_OUTPUT_DIR / "aggregate_top_tokens_scratchpad.png",
    section="scratchpad",
)
render_aggregate_top_tokens_figure(
    rows,
    PNG_OUTPUT_DIR / "aggregate_top_tokens_response.png",
    section="response",
)

# Print the scratchpad summary for quick eyeballing
print("\n=== Scratchpad top-firing tokens per probe (stopwords filtered) ===")
agg = aggregate_top_tokens_across_af(rows, section="scratchpad")
for probe in PROBES:
    print(f"\n{PROBE_LABELS[probe]}:")
    for tok, cnt, mean_score in agg[probe]:
        print(f"  {tok!r:<28}  {cnt:>2} transcripts  (mean score {mean_score:+.3f})")